# Portable execution configuration

This public copy was prepared from the executed JPCO revision notebook.

Machine-specific absolute paths and saved execution outputs were removed.
The original numerical summaries are provided separately in this revision
package.

By default, the notebook treats its current working directory as the
revision root. If required, set the following environment variables before
starting Jupyter:

- `GW_GLITCH_REVISION_ROOT`: directory containing the revision scripts and
  result subdirectories;
- `GW_GLITCH_PROJECT_ROOT`: parent `GW_Glitch` project directory containing
  historical processed datasets;
- `GW_GLITCH_RESEARCH_ROOT`: parent research directory;
- `GW_GLITCH_IMAGE_PROJECT_ROOT`: archived Gravity Spy image-project root.

The full source datasets, trained checkpoints, prediction arrays, and local
execution logs are not stored in Git.


In [ ]:
import os
from pathlib import Path

PORTABLE_REVISION_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_REVISION_ROOT",
        Path.cwd(),
    )
).expanduser().resolve()

PORTABLE_PROJECT_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_PROJECT_ROOT",
        PORTABLE_REVISION_ROOT.parent.parent,
    )
).expanduser().resolve()

PORTABLE_RESEARCH_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_RESEARCH_ROOT",
        PORTABLE_PROJECT_ROOT.parent,
    )
).expanduser().resolve()

PORTABLE_DESKTOP_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_DESKTOP_ROOT",
        PORTABLE_RESEARCH_ROOT.parent,
    )
).expanduser().resolve()

PORTABLE_IMAGE_PROJECT_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_IMAGE_PROJECT_ROOT",
        PORTABLE_RESEARCH_ROOT / "Glitch_Image",
    )
).expanduser().resolve()

print("Portable revision root:", PORTABLE_REVISION_ROOT)
print("Portable project root:", PORTABLE_PROJECT_ROOT)
print("Portable research root:", PORTABLE_RESEARCH_ROOT)
print("Portable image-project root:", PORTABLE_IMAGE_PROJECT_ROOT)


# Feature Provenance and Dataset-Construction Audit

This notebook reconstructs the provenance and preprocessing of the tabular
Gravity Spy dataset used in the JPCO manuscript revision.

The audit identifies the canonical source files, documents the transformation
from observing-run files to the 11-feature and 9-feature datasets, verifies
feature relationships, and records the definitions and units needed for the
revised manuscript.

This stage performs no model training and does not access model predictions.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(
    str(PORTABLE_PROJECT_ROOT)
).resolve()

REVISION_ROOT = PROJECT_ROOT / "revision_2026"

OUTPUT_DIR = (
    REVISION_ROOT
    / "feature_provenance_v1"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Python executable:", sys.executable)
print("Operating system:", platform.platform())
print("Architecture:", platform.machine())
print("Project root:", PROJECT_ROOT)
print("Revision root:", REVISION_ROOT)
print("Output directory:", OUTPUT_DIR)
print()

assert PROJECT_ROOT.is_dir(), (
    f"Project root not found: {PROJECT_ROOT}"
)

assert REVISION_ROOT.is_dir(), (
    f"Revision root not found: {REVISION_ROOT}"
)

assert Path.cwd().resolve() == REVISION_ROOT, (
    "Open the notebook from revision_2026 before continuing.\n"
    f"Current directory: {Path.cwd().resolve()}"
)

print("DIRECTORY CHECK PASSED")

In [ ]:
RAW_SOURCE_NAMES = [
    "H1_O1.csv",
    "H1_O2.csv",
    "H1_O3a.csv",
    "H1_O3b.csv",
    "L1_O1.csv",
    "L1_O2.csv",
    "L1_O3a.csv",
    "L1_O3b.csv",
]

PROCESSED_NAMES = [
    "original_data.csv",
    "sampled_data.csv",
    "original_data_cleaned.csv",
    "sampled_data_cleaned.csv",
]

REVISION_DATA_NAMES = [
    "manifest.json",
    "o3_primary.csv.gz",
    "split_assignments.csv.gz",
    "split_summary.csv",
    "tuning_random_split.csv.gz",
]

TARGET_NAMES = (
    RAW_SOURCE_NAMES
    + PROCESSED_NAMES
)

print("Raw source files expected:", len(RAW_SOURCE_NAMES))
print("Processed datasets expected:", len(PROCESSED_NAMES))
print("Revision datasets expected:", len(REVISION_DATA_NAMES))

In [ ]:
from collections import defaultdict

excluded_directory_names = {
    ".git",
    "__pycache__",
    ".ipynb_checkpoints",
    "targeted_ablation_v1",
    "complexity_benchmark_v1",
    "native_attribution_v1",
}

matches = defaultdict(list)

print("Searching project directory...")
print("This may take a minute if the project contains many copies.")
print()

for path in PROJECT_ROOT.rglob("*"):
    if not path.is_file():
        continue

    relative_parts = path.relative_to(PROJECT_ROOT).parts

    if any(
        part in excluded_directory_names
        for part in relative_parts
    ):
        continue

    if path.name in TARGET_NAMES:
        matches[path.name].append(path.resolve())

for filename in TARGET_NAMES:
    candidates = sorted(
        matches.get(filename, []),
        key=lambda p: str(p).lower(),
    )

    print(filename)

    if not candidates:
        print("  NOT FOUND")
    else:
        for candidate in candidates:
            size_mib = candidate.stat().st_size / (1024 ** 2)
            print(f"  {candidate}")
            print(f"    size: {size_mib:.3f} MiB")

    print()

print("PROJECT SEARCH COMPLETE")

In [ ]:
REVISION_DATA_DIR = (
    REVISION_ROOT
    / "jpco_o3_splits_v1"
)

revision_files = {}

print("Frozen revision dataset:")
print(REVISION_DATA_DIR)
print()

assert REVISION_DATA_DIR.is_dir(), (
    f"Revision dataset directory not found: {REVISION_DATA_DIR}"
)

for filename in REVISION_DATA_NAMES:
    path = REVISION_DATA_DIR / filename

    if path.is_file():
        revision_files[filename] = path.resolve()

        size_mib = path.stat().st_size / (1024 ** 2)

        print(f"FOUND: {filename}")
        print(f"  path: {path}")
        print(f"  size: {size_mib:.3f} MiB")
    else:
        print(f"MISSING: {filename}")

print()
print("REVISION FILE CHECK COMPLETE")

In [ ]:
def read_columns(path):
    if path.suffix.lower() == ".json":
        return None

    try:
        frame = pd.read_csv(
            path,
            nrows=0,
            compression="infer",
        )
        return frame.columns.tolist()
    except Exception as error:
        return {
            "error_type": type(error).__name__,
            "error_message": str(error),
        }


schema_records = []

for filename in TARGET_NAMES:
    for candidate in sorted(
        matches.get(filename, []),
        key=lambda p: str(p).lower(),
    ):
        columns = read_columns(candidate)

        schema_records.append(
            {
                "filename": filename,
                "path": str(candidate),
                "size_bytes": candidate.stat().st_size,
                "size_mib": candidate.stat().st_size / (1024 ** 2),
                "column_count": (
                    len(columns)
                    if isinstance(columns, list)
                    else None
                ),
                "columns": columns,
            }
        )

for filename, path in revision_files.items():
    columns = read_columns(path)

    schema_records.append(
        {
            "filename": filename,
            "path": str(path),
            "size_bytes": path.stat().st_size,
            "size_mib": path.stat().st_size / (1024 ** 2),
            "column_count": (
                len(columns)
                if isinstance(columns, list)
                else None
            ),
            "columns": columns,
        }
    )

print("SCHEMA INSPECTION")
print()

for record in schema_records:
    print(record["filename"])
    print("  path:", record["path"])
    print("  columns:", record["column_count"])
    print("  names:", record["columns"])
    print()

print("SCHEMA INSPECTION COMPLETE")

In [ ]:
raw_schema_records = [
    record
    for record in schema_records
    if record["filename"] in RAW_SOURCE_NAMES
]

raw_schema_groups = {}

for record in raw_schema_records:
    columns = record["columns"]

    if isinstance(columns, list):
        key = tuple(columns)
    else:
        key = json.dumps(columns, sort_keys=True)

    raw_schema_groups.setdefault(key, []).append(
        record["path"]
    )

print(
    "Distinct raw-source schemas:",
    len(raw_schema_groups),
)

for index, (schema, paths) in enumerate(
    raw_schema_groups.items(),
    start=1,
):
    print()
    print(f"RAW SCHEMA {index}")

    if isinstance(schema, tuple):
        print("Columns:", list(schema))
    else:
        print("Schema error:", schema)

    print("Files:")

    for path in paths:
        print(" ", path)

In [ ]:
MANIFEST_PATH = (
    REVISION_DATA_DIR
    / "manifest.json"
)

manifest = json.loads(
    MANIFEST_PATH.read_text()
)

print(json.dumps(manifest, indent=2))

In [ ]:
print("Feature sets recorded in manifest:")
print()

for name, features in manifest["feature_sets"].items():
    print(name)
    print("  count:", len(features))
    print("  features:", features)
    print()

In [ ]:
discovery_report = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "stage": "feature provenance source discovery",
    "completed": True,
    "training_performed": False,
    "model_predictions_accessed": False,
    "test_results_accessed": False,
    "project_root": str(PROJECT_ROOT),
    "revision_root": str(REVISION_ROOT),
    "output_directory": str(OUTPUT_DIR),
    "expected_raw_source_names": RAW_SOURCE_NAMES,
    "expected_processed_names": PROCESSED_NAMES,
    "revision_data_directory": str(
        REVISION_DATA_DIR
    ),
    "matches": {
        filename: [
            str(path)
            for path in sorted(
                matches.get(filename, []),
                key=lambda p: str(p).lower(),
            )
        ]
        for filename in TARGET_NAMES
    },
    "revision_files": {
        filename: str(path)
        for filename, path in revision_files.items()
    },
    "schema_records": schema_records,
    "raw_schema_group_count": len(
        raw_schema_groups
    ),
    "manifest_feature_sets": manifest[
        "feature_sets"
    ],
    "versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
}

REPORT_PATH = (
    OUTPUT_DIR
    / "source_discovery_report.json"
)

REPORT_PATH.write_text(
    json.dumps(
        discovery_report,
        indent=2,
        allow_nan=False,
    )
    + "\n"
)

print("REPORT SAVED:")
print(REPORT_PATH)
print()
print(
    "File size:",
    REPORT_PATH.stat().st_size,
    "bytes",
)
print()
print("SOURCE DISCOVERY COMPLETE")
print("No training or model evaluation was performed.")

In [ ]:
saved_report = json.loads(
    REPORT_PATH.read_text()
)

assert saved_report["completed"] is True
assert saved_report["training_performed"] is False
assert saved_report["model_predictions_accessed"] is False
assert saved_report["test_results_accessed"] is False

assert set(
    saved_report["revision_files"]
) == set(REVISION_DATA_NAMES)

raw_files_found = {
    name: len(
        saved_report["matches"].get(name, [])
    )
    for name in RAW_SOURCE_NAMES
}

processed_files_found = {
    name: len(
        saved_report["matches"].get(name, [])
    )
    for name in PROCESSED_NAMES
}

print("Raw-source candidate counts:")
print(
    json.dumps(
        raw_files_found,
        indent=2,
    )
)

print()
print("Processed-dataset candidate counts:")
print(
    json.dumps(
        processed_files_found,
        indent=2,
    )
)

missing_raw = [
    name
    for name, count in raw_files_found.items()
    if count == 0
]

missing_processed = [
    name
    for name, count
    in processed_files_found.items()
    if count == 0
]

print()

if missing_raw or missing_processed:
    print("DISCOVERY INCOMPLETE")

    if missing_raw:
        print(
            "Missing raw files:",
            missing_raw,
        )

    if missing_processed:
        print(
            "Missing processed files:",
            missing_processed,
        )

    print(
        "Share this output before searching "
        "the external drive."
    )
else:
    print("SOURCE DISCOVERY CHECK PASSED")
    print(
        "All expected raw and processed "
        "dataset names were located."
    )

## Stage 2 — Canonical datasets and observing-run scope

This stage verifies duplicate processed files, inventories the four available
O3 catalogs, and determines whether the frozen manuscript dataset was derived
exclusively from O3a/O3b data.

No files are modified and no model outputs are accessed.

In [ ]:
from pathlib import Path

CANONICAL_RAW = {
    "H1_O3a.csv": (
        PROJECT_ROOT
        / "H1_O3a.csv"
    ),
    "H1_O3b.csv": (
        PROJECT_ROOT
        / "H1_O3b.csv"
    ),
    "L1_O3a.csv": (
        PROJECT_ROOT
        / "Round1"
        / "L1_O3a.csv"
    ),
    "L1_O3b.csv": (
        PROJECT_ROOT
        / "Round1"
        / "L1_O3b.csv"
    ),
}

CANONICAL_PROCESSED = {
    "original_data.csv": (
        PROJECT_ROOT
        / "original_data.csv"
    ),
    "sampled_data.csv": (
        PROJECT_ROOT
        / "sampled_data.csv"
    ),
    "original_data_cleaned.csv": (
        PROJECT_ROOT
        / "original_data_cleaned.csv"
    ),
    "sampled_data_cleaned.csv": (
        PROJECT_ROOT
        / "sampled_data_cleaned.csv"
    ),
}

DUPLICATE_PROCESSED = {
    "original_data_cleaned.csv": (
        PROJECT_ROOT
        / "glitch-tabular-prd"
        / "data"
        / "processed"
        / "original_data_cleaned.csv"
    ),
    "sampled_data_cleaned.csv": (
        PROJECT_ROOT
        / "glitch-tabular-prd"
        / "data"
        / "processed"
        / "sampled_data_cleaned.csv"
    ),
}

for group_name, mapping in [
    ("Raw O3 catalogs", CANONICAL_RAW),
    ("Canonical processed datasets", CANONICAL_PROCESSED),
    ("Repository processed copies", DUPLICATE_PROCESSED),
]:
    print(group_name)

    for name, path in mapping.items():
        assert path.is_file(), f"Missing: {path}"

        print(
            f"  {name}: "
            f"{path.stat().st_size / 1024**2:.3f} MiB"
        )

    print()

print("CANONICAL PATH CHECK PASSED")

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


processed_hashes = {}

for name, path in CANONICAL_PROCESSED.items():
    print("Hashing:", path)

    processed_hashes[
        f"canonical::{name}"
    ] = sha256_file(path)

for name, path in DUPLICATE_PROCESSED.items():
    print("Hashing:", path)

    processed_hashes[
        f"repository_copy::{name}"
    ] = sha256_file(path)

print()
print("Processed dataset hashes:")

for key, value in processed_hashes.items():
    print(f"{key}")
    print(f"  {value}")

for name in DUPLICATE_PROCESSED:
    canonical_hash = processed_hashes[
        f"canonical::{name}"
    ]

    duplicate_hash = processed_hashes[
        f"repository_copy::{name}"
    ]

    assert canonical_hash == duplicate_hash, (
        f"The two copies of {name} differ."
    )

print()
print("DUPLICATE HASH CHECK PASSED")
print(
    "The repository and project-root cleaned "
    "datasets are byte-identical."
)

In [ ]:
MANIFEST_PATH = (
    REVISION_DATA_DIR
    / "manifest.json"
)

manifest = json.loads(
    MANIFEST_PATH.read_text()
)

print(json.dumps(manifest, indent=2))

In [ ]:
print("Manifest top-level fields:")
print()

for key in manifest:
    value = manifest[key]

    if isinstance(value, dict):
        description = (
            f"dictionary with {len(value)} entries"
        )
    elif isinstance(value, list):
        description = (
            f"list with {len(value)} entries"
        )
    else:
        description = repr(value)

    print(f"{key}: {description}")

In [ ]:
PRIMARY_PATH = (
    REVISION_DATA_DIR
    / "o3_primary.csv.gz"
)

PROVENANCE_COLUMNS = [
    "source_record_id",
    "source_catalog",
    "source_row",
    "ifo",
    "gravityspy_id",
    "peak_gps_ns",
    "ml_label",
]

primary_provenance = pd.read_csv(
    PRIMARY_PATH,
    usecols=PROVENANCE_COLUMNS,
)

print("Frozen primary rows:", len(primary_provenance))
print(
    "Unique source_record_id:",
    primary_provenance[
        "source_record_id"
    ].nunique(),
)
print(
    "Unique source catalogs:",
    primary_provenance[
        "source_catalog"
    ].nunique(),
)

assert (
    primary_provenance["source_record_id"].is_unique
)

print()
print("Source-catalog counts:")

catalog_counts = (
    primary_provenance
    .groupby(
        ["source_catalog", "ifo"],
        dropna=False,
    )
    .agg(
        rows=("source_record_id", "size"),
        unique_source_rows=(
            "source_row",
            "nunique",
        ),
        minimum_source_row=(
            "source_row",
            "min",
        ),
        maximum_source_row=(
            "source_row",
            "max",
        ),
        classes=("ml_label", "nunique"),
        minimum_peak_gps_ns=(
            "peak_gps_ns",
            "min",
        ),
        maximum_peak_gps_ns=(
            "peak_gps_ns",
            "max",
        ),
    )
    .reset_index()
)

display(catalog_counts)

In [ ]:
observed_catalogs = set(
    primary_provenance[
        "source_catalog"
    ].astype(str)
)

print("Observed catalog labels:")

for catalog in sorted(observed_catalogs):
    print(" ", catalog)

In [ ]:
catalog_text = " ".join(
    sorted(observed_catalogs)
).lower()

contains_o1 = "o1" in catalog_text
contains_o2 = "o2" in catalog_text
contains_o3a = "o3a" in catalog_text
contains_o3b = "o3b" in catalog_text

print("Catalog-label indicators:")
print("  O1 present:", contains_o1)
print("  O2 present:", contains_o2)
print("  O3a present:", contains_o3a)
print("  O3b present:", contains_o3b)
print()

if (
    not contains_o1
    and not contains_o2
    and contains_o3a
    and contains_o3b
):
    print("FROZEN COHORT IS EXPLICITLY O3a/O3b")
elif contains_o1 or contains_o2:
    print(
        "O1/O2 APPEAR IN FROZEN PROVENANCE — "
        "SHARE OUTPUT BEFORE CONTINUING"
    )
else:
    print(
        "CATALOG LABELS DO NOT ENCODE RUN NAMES "
        "CLEARLY — ROW-LEVEL MAPPING IS REQUIRED"
    )

In [ ]:
RAW_COLUMNS = [
    "event_time",
    "ifo",
    "peak_time",
    "peak_time_ns",
    "start_time",
    "start_time_ns",
    "duration",
    "peak_frequency",
    "central_freq",
    "bandwidth",
    "amplitude",
    "snr",
    "q_value",
    "gravityspy_id",
    "ml_label",
    "ml_confidence",
]

raw_catalog_records = []

for catalog_name, path in CANONICAL_RAW.items():
    print("Inspecting:", catalog_name)

    rows = 0
    missing_label_rows = 0
    detector_values = set()
    label_values = set()
    minimum_peak_time = None
    maximum_peak_time = None
    minimum_peak_time_ns = None
    maximum_peak_time_ns = None

    for chunk in pd.read_csv(
        path,
        usecols=RAW_COLUMNS,
        chunksize=100_000,
        float_precision="round_trip",
    ):
        rows += len(chunk)

        missing_label_rows += int(
            chunk["ml_label"].isna().sum()
        )

        detector_values.update(
            chunk["ifo"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        label_values.update(
            chunk["ml_label"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        chunk_peak_min = chunk[
            "peak_time"
        ].min()

        chunk_peak_max = chunk[
            "peak_time"
        ].max()

        chunk_ns_min = chunk[
            "peak_time_ns"
        ].min()

        chunk_ns_max = chunk[
            "peak_time_ns"
        ].max()

        if pd.notna(chunk_peak_min):
            minimum_peak_time = (
                chunk_peak_min
                if minimum_peak_time is None
                else min(
                    minimum_peak_time,
                    chunk_peak_min,
                )
            )

        if pd.notna(chunk_peak_max):
            maximum_peak_time = (
                chunk_peak_max
                if maximum_peak_time is None
                else max(
                    maximum_peak_time,
                    chunk_peak_max,
                )
            )

        if pd.notna(chunk_ns_min):
            minimum_peak_time_ns = (
                chunk_ns_min
                if minimum_peak_time_ns is None
                else min(
                    minimum_peak_time_ns,
                    chunk_ns_min,
                )
            )

        if pd.notna(chunk_ns_max):
            maximum_peak_time_ns = (
                chunk_ns_max
                if maximum_peak_time_ns is None
                else max(
                    maximum_peak_time_ns,
                    chunk_ns_max,
                )
            )

    record = {
        "catalog": catalog_name,
        "path": str(path),
        "rows": rows,
        "detectors": sorted(detector_values),
        "classes": len(label_values),
        "missing_ml_label_rows": (
            missing_label_rows
        ),
        "minimum_peak_time": (
            None
            if minimum_peak_time is None
            else float(minimum_peak_time)
        ),
        "maximum_peak_time": (
            None
            if maximum_peak_time is None
            else float(maximum_peak_time)
        ),
        "minimum_peak_time_ns": (
            None
            if minimum_peak_time_ns is None
            else int(minimum_peak_time_ns)
        ),
        "maximum_peak_time_ns": (
            None
            if maximum_peak_time_ns is None
            else int(maximum_peak_time_ns)
        ),
    }

    raw_catalog_records.append(record)

    print(json.dumps(record, indent=2))
    print()

raw_catalog_summary = pd.DataFrame(
    raw_catalog_records
)

display(raw_catalog_summary)

In [ ]:
PROCESSED_FEATURE_COLUMNS = [
    "peak_time",
    "peak_time_ns",
    "start_time",
    "start_time_ns",
    "duration",
    "peak_frequency",
    "central_freq",
    "bandwidth",
    "amplitude",
    "snr",
    "q_value",
]

processed_records = []

for dataset_name, path in (
    CANONICAL_PROCESSED.items()
):
    print("Inspecting:", dataset_name)

    rows = 0
    label_values = set()
    missing_by_column = None

    for chunk in pd.read_csv(
        path,
        chunksize=100_000,
        float_precision="round_trip",
    ):
        rows += len(chunk)

        label_values.update(
            chunk["ml_label"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        chunk_missing = (
            chunk.isna()
            .sum()
            .astype(int)
        )

        if missing_by_column is None:
            missing_by_column = chunk_missing
        else:
            missing_by_column = (
                missing_by_column.add(
                    chunk_missing,
                    fill_value=0,
                )
            )

    record = {
        "dataset": dataset_name,
        "path": str(path),
        "rows": rows,
        "columns": (
            pd.read_csv(
                path,
                nrows=0,
            )
            .columns
            .tolist()
        ),
        "classes": len(label_values),
        "labels": sorted(label_values),
        "missing_by_column": {
            column: int(value)
            for column, value
            in missing_by_column.items()
        },
    }

    processed_records.append(record)

    print("  rows:", record["rows"])
    print("  classes:", record["classes"])
    print(
        "  nonzero missing counts:",
        {
            key: value
            for key, value
            in record[
                "missing_by_column"
            ].items()
            if value != 0
        },
    )
    print()

processed_summary = pd.DataFrame(
    [
        {
            "dataset": record["dataset"],
            "rows": record["rows"],
            "columns": len(
                record["columns"]
            ),
            "classes": record["classes"],
        }
        for record in processed_records
    ]
)

display(processed_summary)

In [ ]:
original_columns = (
    pd.read_csv(
        CANONICAL_PROCESSED[
            "original_data.csv"
        ],
        nrows=0,
    )
    .columns
    .tolist()
)

cleaned_columns = (
    pd.read_csv(
        CANONICAL_PROCESSED[
            "original_data_cleaned.csv"
        ],
        nrows=0,
    )
    .columns
    .tolist()
)

removed_columns = [
    column
    for column in original_columns
    if column not in cleaned_columns
]

added_columns = [
    column
    for column in cleaned_columns
    if column not in original_columns
]

print("Original columns:")
print(original_columns)

print()
print("Cleaned columns:")
print(cleaned_columns)

print()
print("Removed columns:")
print(removed_columns)

print()
print("Added columns:")
print(added_columns)

assert removed_columns == [
    "start_time",
    "bandwidth",
]

assert added_columns == []

print()
print("11-TO-9 SCHEMA CHECK PASSED")
print(
    "The cleaning step removed only "
    "start_time and bandwidth."
)

In [ ]:
stage2_report = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "stage": (
        "canonical dataset and "
        "observing-run scope audit"
    ),
    "completed": True,
    "training_performed": False,
    "model_predictions_accessed": False,
    "test_results_accessed": False,
    "canonical_raw_files": {
        name: str(path)
        for name, path
        in CANONICAL_RAW.items()
    },
    "canonical_processed_files": {
        name: str(path)
        for name, path
        in CANONICAL_PROCESSED.items()
    },
    "duplicate_processed_files": {
        name: str(path)
        for name, path
        in DUPLICATE_PROCESSED.items()
    },
    "processed_sha256": (
        processed_hashes
    ),
    "frozen_primary_rows": int(
        len(primary_provenance)
    ),
    "observed_source_catalogs": sorted(
        observed_catalogs
    ),
    "catalog_counts": (
        catalog_counts
        .to_dict(orient="records")
    ),
    "catalog_label_indicators": {
        "o1_present": contains_o1,
        "o2_present": contains_o2,
        "o3a_present": contains_o3a,
        "o3b_present": contains_o3b,
    },
    "raw_catalog_summary": (
        raw_catalog_records
    ),
    "processed_dataset_summary": (
        processed_records
    ),
    "schema_transformation": {
        "original_columns": (
            original_columns
        ),
        "cleaned_columns": (
            cleaned_columns
        ),
        "removed_columns": (
            removed_columns
        ),
        "added_columns": (
            added_columns
        ),
    },
    "manifest_feature_sets": manifest[
        "feature_sets"
    ],
}

STAGE2_REPORT_PATH = (
    OUTPUT_DIR
    / "canonical_scope_report.json"
)

STAGE2_REPORT_PATH.write_text(
    json.dumps(
        stage2_report,
        indent=2,
        allow_nan=False,
    )
    + "\n"
)

print("REPORT SAVED:")
print(STAGE2_REPORT_PATH)
print()
print(
    "File size:",
    STAGE2_REPORT_PATH.stat().st_size,
    "bytes",
)
print()
print("STAGE 2 COMPLETE")

In [ ]:
saved_stage2 = json.loads(
    STAGE2_REPORT_PATH.read_text()
)

assert saved_stage2["completed"] is True
assert (
    saved_stage2[
        "training_performed"
    ]
    is False
)
assert (
    saved_stage2[
        "model_predictions_accessed"
    ]
    is False
)
assert (
    saved_stage2[
        "test_results_accessed"
    ]
    is False
)

assert saved_stage2[
    "schema_transformation"
]["removed_columns"] == [
    "start_time",
    "bandwidth",
]

assert saved_stage2[
    "schema_transformation"
]["added_columns"] == []

print("Observed frozen source catalogs:")

for catalog in saved_stage2[
    "observed_source_catalogs"
]:
    print(" ", catalog)

print()
print("Raw catalog summary:")
display(
    pd.DataFrame(
        saved_stage2[
            "raw_catalog_summary"
        ]
    )
)

print()
print("Processed dataset summary:")
display(
    pd.DataFrame(
        [
            {
                "dataset": record[
                    "dataset"
                ],
                "rows": record["rows"],
                "columns": len(
                    record["columns"]
                ),
                "classes": record[
                    "classes"
                ],
            }
            for record in saved_stage2[
                "processed_dataset_summary"
            ]
        ]
    )
)

print()
print("CANONICAL-SCOPE AUDIT PASSED")
print(
    "Share canonical_scope_report.json "
    "before running correlation or "
    "row-reconstruction analyses."
)

## Stage 3 — Row reconstruction and feature relationships

This stage verifies that the processed datasets were constructed from the four
O3 catalogs, reconstructs the 11-to-9 transformation, validates the sampled
dataset as a subset, and measures the empirical relationships among timestamps,
duration, bandwidth, frequency, and Q.

No model training or model-result access is performed.

In [ ]:
expected_raw_hashes = manifest[
    "source_sha256"
]

observed_raw_hashes = {}

for catalog_name, path in (
    CANONICAL_RAW.items()
):
    print("Hashing:", catalog_name)

    observed_hash = sha256_file(path)

    observed_raw_hashes[
        catalog_name
    ] = observed_hash

    print("  observed:", observed_hash)
    print(
        "  expected:",
        expected_raw_hashes[
            catalog_name
        ],
    )

    assert (
        observed_hash
        == expected_raw_hashes[
            catalog_name
        ]
    ), f"Raw source changed: {catalog_name}"

print()
print("RAW SOURCE HASHES VERIFIED")

In [ ]:
ENRICHED_COLUMNS = [
    "peak_time",
    "peak_time_ns",
    "start_time",
    "start_time_ns",
    "duration",
    "peak_frequency",
    "central_freq",
    "bandwidth",
    "amplitude",
    "snr",
    "q_value",
    "ml_label",
]

raw_frames = {}

for catalog_name, path in (
    CANONICAL_RAW.items()
):
    print("Loading:", catalog_name)

    frame = pd.read_csv(
        path,
        usecols=ENRICHED_COLUMNS,
        float_precision="round_trip",
    )

    frame = frame[
        ENRICHED_COLUMNS
    ].copy()

    raw_frames[catalog_name] = frame

    print(
        f"  rows: {len(frame):,}; "
        f"columns: {len(frame.columns)}"
    )

raw_concatenated = pd.concat(
    [
        raw_frames["H1_O3a.csv"],
        raw_frames["H1_O3b.csv"],
        raw_frames["L1_O3a.csv"],
        raw_frames["L1_O3b.csv"],
    ],
    ignore_index=True,
)

print()
print(
    "Concatenated raw rows:",
    f"{len(raw_concatenated):,}",
)

assert len(raw_concatenated) == 500_524
assert not raw_concatenated.isna().any().any()

print("RAW COLUMN LOAD PASSED")

In [ ]:
original_data = pd.read_csv(
    CANONICAL_PROCESSED[
        "original_data.csv"
    ],
    float_precision="round_trip",
)

assert (
    original_data.columns.tolist()
    == ENRICHED_COLUMNS
)

assert len(original_data) == 500_524

same_order_exact = (
    raw_concatenated.equals(
        original_data
    )
)

print(
    "Exact equality in concatenation order:",
    same_order_exact,
)

In [ ]:
raw_row_hashes = (
    pd.util.hash_pandas_object(
        raw_concatenated,
        index=False,
        categorize=True,
    )
)

processed_row_hashes = (
    pd.util.hash_pandas_object(
        original_data,
        index=False,
        categorize=True,
    )
)

raw_hash_counts = (
    raw_row_hashes
    .value_counts()
    .sort_index()
)

processed_hash_counts = (
    processed_row_hashes
    .value_counts()
    .sort_index()
)

same_row_multiset = (
    raw_hash_counts.equals(
        processed_hash_counts
    )
)

print(
    "Same rows including duplicate multiplicities:",
    same_row_multiset,
)

assert same_row_multiset, (
    "original_data.csv is not an exact "
    "row multiset of the four O3 catalogs."
)

print()
print("ORIGINAL DATASET RECONSTRUCTION PASSED")

if same_order_exact:
    print(
        "The processed dataset preserves the "
        "tested catalog concatenation order."
    )
else:
    print(
        "The processed dataset contains exactly "
        "the same rows in a different order."
    )

In [ ]:
dtype_comparison = pd.DataFrame(
    {
        "raw_concatenated": (
            raw_concatenated.dtypes.astype(str)
        ),
        "original_data": (
            original_data.dtypes.astype(str)
        ),
    }
)

dtype_comparison[
    "same_dtype"
] = (
    dtype_comparison[
        "raw_concatenated"
    ]
    == dtype_comparison[
        "original_data"
    ]
)

display(dtype_comparison)

In [ ]:
print("Columns with different inferred dtypes:")

different_dtypes = (
    dtype_comparison.loc[
        ~dtype_comparison[
            "same_dtype"
        ]
    ]
)

if len(different_dtypes) == 0:
    print("  None")
else:
    display(different_dtypes)

In [ ]:
TIME_COMPONENT_COLUMNS = [
    "peak_time",
    "peak_time_ns",
    "start_time",
    "start_time_ns",
]

for column in TIME_COMPONENT_COLUMNS:
    print(column)

    for dataset_name, frame in [
        (
            "raw_concatenated",
            raw_concatenated,
        ),
        (
            "original_data",
            original_data,
        ),
    ]:
        values = pd.to_numeric(
            frame[column],
            errors="raise",
        ).to_numpy(dtype=np.float64)

        maximum_distance_from_integer = float(
            np.max(
                np.abs(
                    values
                    - np.rint(values)
                )
            )
        )

        print(
            f"  {dataset_name}: "
            f"dtype={frame[column].dtype}; "
            f"maximum distance from integer="
            f"{maximum_distance_from_integer:.17g}"
        )

    print()

In [ ]:
INTEGER_COLUMNS = [
    "peak_time",
    "peak_time_ns",
    "start_time",
    "start_time_ns",
]

FLOAT_COLUMNS = [
    "duration",
    "peak_frequency",
    "central_freq",
    "bandwidth",
    "amplitude",
    "snr",
    "q_value",
]


def normalize_dataset(frame):
    normalized = frame[
        ENRICHED_COLUMNS
    ].copy()

    for column in INTEGER_COLUMNS:
        numeric = pd.to_numeric(
            normalized[column],
            errors="raise",
        ).to_numpy(dtype=np.float64)

        assert np.isfinite(numeric).all()

        distance_from_integer = np.abs(
            numeric
            - np.rint(numeric)
        )

        assert (
            distance_from_integer.max()
            == 0
        ), (
            f"{column} contains a "
            "non-integer value."
        )

        normalized[column] = (
            np.rint(numeric)
            .astype(np.int64)
        )

    for column in FLOAT_COLUMNS:
        normalized[column] = (
            pd.to_numeric(
                normalized[column],
                errors="raise",
            )
            .astype(np.float64)
        )

        assert np.isfinite(
            normalized[column]
        ).all()

    normalized["ml_label"] = (
        normalized["ml_label"]
        .astype(str)
    )

    return normalized


raw_normalized = normalize_dataset(
    raw_concatenated
)

processed_normalized = normalize_dataset(
    original_data
)

print("Normalized raw dtypes:")
print(raw_normalized.dtypes)

print()
print("Normalized processed dtypes:")
print(processed_normalized.dtypes)

In [ ]:
same_order_after_normalization = (
    raw_normalized.equals(
        processed_normalized
    )
)

print(
    "Exact equality in concatenation order "
    "after dtype normalization:",
    same_order_after_normalization,
)

In [ ]:
RECORD_KEY_COLUMNS = [
    "peak_time",
    "peak_time_ns",
    "start_time",
    "start_time_ns",
    "ml_label",
]

raw_key_hashes = (
    pd.util.hash_pandas_object(
        raw_normalized[
            RECORD_KEY_COLUMNS
        ],
        index=False,
        categorize=True,
    )
)

processed_key_hashes = (
    pd.util.hash_pandas_object(
        processed_normalized[
            RECORD_KEY_COLUMNS
        ],
        index=False,
        categorize=True,
    )
)

raw_key_counts = (
    raw_key_hashes
    .value_counts()
    .sort_index()
)

processed_key_counts = (
    processed_key_hashes
    .value_counts()
    .sort_index()
)

same_identity_key_multiset = (
    raw_key_counts.equals(
        processed_key_counts
    )
)

print(
    "Same record-identity keys including "
    "duplicate multiplicities:",
    same_identity_key_multiset,
)

print(
    "Raw unique identity keys:",
    raw_key_hashes.nunique(),
)

print(
    "Processed unique identity keys:",
    processed_key_hashes.nunique(),
)

print(
    "Raw duplicated-key rows:",
    int(
        raw_key_hashes.duplicated(
            keep=False
        ).sum()
    ),
)

print(
    "Processed duplicated-key rows:",
    int(
        processed_key_hashes.duplicated(
            keep=False
        ).sum()
    ),
)

In [ ]:
raw_for_alignment = (
    raw_normalized.copy()
)

processed_for_alignment = (
    processed_normalized.copy()
)

raw_for_alignment[
    "_raw_row"
] = np.arange(
    len(raw_for_alignment),
    dtype=np.int64,
)

processed_for_alignment[
    "_processed_row"
] = np.arange(
    len(processed_for_alignment),
    dtype=np.int64,
)

raw_for_alignment[
    "_duplicate_occurrence"
] = (
    raw_for_alignment
    .groupby(
        RECORD_KEY_COLUMNS,
        sort=False,
        dropna=False,
    )
    .cumcount()
)

processed_for_alignment[
    "_duplicate_occurrence"
] = (
    processed_for_alignment
    .groupby(
        RECORD_KEY_COLUMNS,
        sort=False,
        dropna=False,
    )
    .cumcount()
)

ALIGNMENT_KEYS = (
    RECORD_KEY_COLUMNS
    + ["_duplicate_occurrence"]
)

aligned = raw_for_alignment.merge(
    processed_for_alignment,
    on=ALIGNMENT_KEYS,
    how="outer",
    suffixes=(
        "_raw",
        "_processed",
    ),
    indicator=True,
    validate="one_to_one",
)

alignment_counts = (
    aligned["_merge"]
    .value_counts(dropna=False)
    .to_dict()
)

print("Alignment counts:")
print(
    json.dumps(
        {
            str(key): int(value)
            for key, value
            in alignment_counts.items()
        },
        indent=2,
    )
)

unmatched_rows = int(
    (
        aligned["_merge"]
        != "both"
    ).sum()
)

print()
print(
    "Unmatched rows after identity alignment:",
    unmatched_rows,
)

In [ ]:
matched = (
    aligned.loc[
        aligned["_merge"].eq("both")
    ]
    .copy()
)

numeric_difference_records = []

for column in FLOAT_COLUMNS:
    raw_values = (
        matched[
            f"{column}_raw"
        ]
        .to_numpy(dtype=np.float64)
    )

    processed_values = (
        matched[
            f"{column}_processed"
        ]
        .to_numpy(dtype=np.float64)
    )

    absolute_error = np.abs(
        raw_values
        - processed_values
    )

    scale = np.maximum(
        np.maximum(
            np.abs(raw_values),
            np.abs(processed_values),
        ),
        np.finfo(np.float64).tiny,
    )

    relative_error = (
        absolute_error
        / scale
    )

    record = {
        "column": column,
        "exactly_equal_rows": int(
            (
                raw_values
                == processed_values
            ).sum()
        ),
        "different_rows": int(
            (
                raw_values
                != processed_values
            ).sum()
        ),
        "maximum_absolute_error": float(
            absolute_error.max()
        ),
        "mean_absolute_error": float(
            absolute_error.mean()
        ),
        "maximum_relative_error": float(
            relative_error.max()
        ),
        "mean_relative_error": float(
            relative_error.mean()
        ),
        "allclose_rtol_1e_minus_12": bool(
            np.allclose(
                raw_values,
                processed_values,
                rtol=1e-12,
                atol=0.0,
            )
        ),
        "allclose_rtol_1e_minus_10": bool(
            np.allclose(
                raw_values,
                processed_values,
                rtol=1e-10,
                atol=0.0,
            )
        ),
    }

    numeric_difference_records.append(
        record
    )

numeric_difference_table = pd.DataFrame(
    numeric_difference_records
)

display(numeric_difference_table)

In [ ]:
RECONSTRUCTION_DIAGNOSTIC_PATH = (
    OUTPUT_DIR
    / "raw_processed_reconstruction_diagnostic.csv"
)

numeric_difference_table.to_csv(
    RECONSTRUCTION_DIAGNOSTIC_PATH,
    index=False,
)

reconstruction_diagnostic = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "raw_rows": int(
        len(raw_normalized)
    ),
    "processed_rows": int(
        len(processed_normalized)
    ),
    "same_order_before_normalization": bool(
        same_order_exact
    ),
    "same_order_after_normalization": bool(
        same_order_after_normalization
    ),
    "same_identity_key_multiset": bool(
        same_identity_key_multiset
    ),
    "alignment_counts": {
        str(key): int(value)
        for key, value
        in alignment_counts.items()
    },
    "unmatched_rows": int(
        unmatched_rows
    ),
    "numeric_differences_file": str(
        RECONSTRUCTION_DIAGNOSTIC_PATH
    ),
}

RECONSTRUCTION_DIAGNOSTIC_JSON = (
    OUTPUT_DIR
    / "raw_processed_reconstruction_diagnostic.json"
)

RECONSTRUCTION_DIAGNOSTIC_JSON.write_text(
    json.dumps(
        reconstruction_diagnostic,
        indent=2,
        allow_nan=False,
    )
    + "\n"
)

print("SAVED:")
print(RECONSTRUCTION_DIAGNOSTIC_PATH)
print(RECONSTRUCTION_DIAGNOSTIC_JSON)
print()
print("RECONSTRUCTION DIAGNOSTIC COMPLETE")

In [ ]:
assert same_identity_key_multiset is True
assert unmatched_rows == 0

assert (
    numeric_difference_table[
        "allclose_rtol_1e_minus_12"
    ].all()
)

reconstruction_tolerance = {
    "relative_tolerance": 1e-12,
    "absolute_tolerance": 0.0,
    "justification": (
        "All 500,524 record-identity keys and "
        "duplicate multiplicities match. Zero "
        "records are unmatched. The largest "
        "observed relative discrepancy is "
        "approximately 1.00103e-13 and results "
        "from CSV floating-point serialization."
    ),
}

raw_processed_reconstruction_passed = True

print("RAW-TO-PROCESSED RECONSTRUCTION PASSED")
print()
print(
    "Matched records:",
    int(
        alignment_counts.get(
            "both",
            0,
        )
    ),
)
print(
    "Unmatched records:",
    unmatched_rows,
)
print(
    "Adopted relative tolerance:",
    reconstruction_tolerance[
        "relative_tolerance"
    ],
)
print()
print(
    reconstruction_tolerance[
        "justification"
    ]
)

In [ ]:
FINAL_RECONSTRUCTION_PATH = (
    OUTPUT_DIR
    / "raw_processed_reconstruction_final.json"
)

final_reconstruction = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "completed": True,
    "raw_rows": int(
        len(raw_normalized)
    ),
    "processed_rows": int(
        len(processed_normalized)
    ),
    "same_record_identity_multiset": bool(
        same_identity_key_multiset
    ),
    "matched_rows": int(
        alignment_counts.get(
            "both",
            0,
        )
    ),
    "raw_only_rows": int(
        alignment_counts.get(
            "left_only",
            0,
        )
    ),
    "processed_only_rows": int(
        alignment_counts.get(
            "right_only",
            0,
        )
    ),
    "relative_tolerance": 1e-12,
    "absolute_tolerance": 0.0,
    "all_numeric_columns_within_tolerance": bool(
        numeric_difference_table[
            "allclose_rtol_1e_minus_12"
        ].all()
    ),
    "maximum_observed_relative_error": float(
        numeric_difference_table[
            "maximum_relative_error"
        ].max()
    ),
    "order_preserved": bool(
        same_order_after_normalization
    ),
    "conclusion": (
        "The processed 11-feature dataset "
        "contains all 500,524 raw O3 records "
        "with identical record-identity "
        "multiplicities. All numeric values "
        "agree within rtol=1e-12. The observed "
        "differences are limited to row ordering "
        "and negligible CSV floating-point "
        "round-trip effects in duration and "
        "amplitude."
    ),
}

FINAL_RECONSTRUCTION_PATH.write_text(
    json.dumps(
        final_reconstruction,
        indent=2,
        allow_nan=False,
    )
    + "\n"
)

print("SAVED:")
print(FINAL_RECONSTRUCTION_PATH)
print()
print(
    final_reconstruction[
        "conclusion"
    ]
)

In [ ]:
CLEANED_COLUMNS = [
    "peak_time",
    "peak_time_ns",
    "start_time_ns",
    "duration",
    "peak_frequency",
    "central_freq",
    "amplitude",
    "snr",
    "q_value",
    "ml_label",
]

CLEANED_INTEGER_COLUMNS = [
    "peak_time",
    "peak_time_ns",
    "start_time_ns",
]

CLEANED_FLOAT_COLUMNS = [
    "duration",
    "peak_frequency",
    "central_freq",
    "amplitude",
    "snr",
    "q_value",
]


def normalize_cleaned_dataset(frame):
    normalized = frame[
        CLEANED_COLUMNS
    ].copy()

    for column in CLEANED_INTEGER_COLUMNS:
        values = pd.to_numeric(
            normalized[column],
            errors="raise",
        ).to_numpy(dtype=np.float64)

        assert np.isfinite(values).all()

        distance_from_integer = np.abs(
            values - np.rint(values)
        )

        assert distance_from_integer.max() == 0

        normalized[column] = (
            np.rint(values)
            .astype(np.int64)
        )

    for column in CLEANED_FLOAT_COLUMNS:
        normalized[column] = (
            pd.to_numeric(
                normalized[column],
                errors="raise",
            )
            .astype(np.float64)
        )

        assert np.isfinite(
            normalized[column]
        ).all()

    normalized["ml_label"] = (
        normalized["ml_label"]
        .astype(str)
    )

    return normalized

In [ ]:
cleaned_data = pd.read_csv(
    CANONICAL_PROCESSED[
        "original_data_cleaned.csv"
    ],
    float_precision="round_trip",
)

assert (
    cleaned_data.columns.tolist()
    == CLEANED_COLUMNS
)

assert len(cleaned_data) == 500_524

cleaned_normalized = (
    normalize_cleaned_dataset(
        cleaned_data
    )
)

expected_cleaned_normalized = (
    processed_normalized
    .drop(
        columns=[
            "start_time",
            "bandwidth",
        ]
    )
    [CLEANED_COLUMNS]
    .copy()
)

print(
    "Expected cleaned rows:",
    len(expected_cleaned_normalized),
)

print(
    "Observed cleaned rows:",
    len(cleaned_normalized),
)

In [ ]:
cleaned_exact_columns = (
    CLEANED_INTEGER_COLUMNS
    + ["ml_label"]
)

cleaned_exact_column_results = {}

for column in cleaned_exact_columns:
    exact = bool(
        expected_cleaned_normalized[
            column
        ].equals(
            cleaned_normalized[
                column
            ]
        )
    )

    cleaned_exact_column_results[
        column
    ] = exact

print("Exact retained-column checks:")
print(
    json.dumps(
        cleaned_exact_column_results,
        indent=2,
    )
)

assert all(
    cleaned_exact_column_results.values()
)

In [ ]:
cleaned_numeric_records = []

for column in CLEANED_FLOAT_COLUMNS:
    expected_values = (
        expected_cleaned_normalized[
            column
        ]
        .to_numpy(dtype=np.float64)
    )

    observed_values = (
        cleaned_normalized[
            column
        ]
        .to_numpy(dtype=np.float64)
    )

    absolute_error = np.abs(
        expected_values
        - observed_values
    )

    scale = np.maximum(
        np.maximum(
            np.abs(expected_values),
            np.abs(observed_values),
        ),
        np.finfo(np.float64).tiny,
    )

    relative_error = (
        absolute_error / scale
    )

    cleaned_numeric_records.append(
        {
            "column": column,
            "exactly_equal_rows": int(
                (
                    expected_values
                    == observed_values
                ).sum()
            ),
            "different_rows": int(
                (
                    expected_values
                    != observed_values
                ).sum()
            ),
            "maximum_absolute_error": float(
                absolute_error.max()
            ),
            "maximum_relative_error": float(
                relative_error.max()
            ),
            "within_rtol_1e_minus_12": bool(
                np.allclose(
                    expected_values,
                    observed_values,
                    rtol=1e-12,
                    atol=0.0,
                )
            ),
        }
    )

cleaned_numeric_table = pd.DataFrame(
    cleaned_numeric_records
)

display(cleaned_numeric_table)

assert cleaned_numeric_table[
    "within_rtol_1e_minus_12"
].all()

cleaned_transformation_passed = True

print()
print("11-TO-9 TRANSFORMATION VERIFIED")
print(
    "Only start_time and bandwidth "
    "were removed; all retained values "
    "are preserved within rtol=1e-12."
)

In [ ]:
CLEANED_DIAGNOSTIC_PATH = (
    OUTPUT_DIR
    / "cleaned_transformation_diagnostic.csv"
)

cleaned_numeric_table.to_csv(
    CLEANED_DIAGNOSTIC_PATH,
    index=False,
)

print("SAVED:", CLEANED_DIAGNOSTIC_PATH)

In [ ]:
sampled_data = pd.read_csv(
    CANONICAL_PROCESSED[
        "sampled_data.csv"
    ],
    float_precision="round_trip",
)

sampled_cleaned = pd.read_csv(
    CANONICAL_PROCESSED[
        "sampled_data_cleaned.csv"
    ],
    float_precision="round_trip",
)

assert (
    sampled_data.columns.tolist()
    == ENRICHED_COLUMNS
)

assert (
    sampled_cleaned.columns.tolist()
    == CLEANED_COLUMNS
)

assert len(sampled_data) == 50_554
assert len(sampled_cleaned) == 50_554

sampled_normalized = normalize_dataset(
    sampled_data
)

sampled_cleaned_normalized = (
    normalize_cleaned_dataset(
        sampled_cleaned
    )
)

expected_sampled_cleaned = (
    sampled_normalized
    .drop(
        columns=[
            "start_time",
            "bandwidth",
        ]
    )
    [CLEANED_COLUMNS]
    .copy()
)

print(
    "Sampled rows:",
    f"{len(sampled_normalized):,}",
)

In [ ]:
sampled_cleaning_records = []

for column in CLEANED_COLUMNS:
    expected_values = (
        expected_sampled_cleaned[
            column
        ]
    )

    observed_values = (
        sampled_cleaned_normalized[
            column
        ]
    )

    if column in CLEANED_FLOAT_COLUMNS:
        expected_array = (
            expected_values.to_numpy(
                dtype=np.float64
            )
        )

        observed_array = (
            observed_values.to_numpy(
                dtype=np.float64
            )
        )

        absolute_error = np.abs(
            expected_array
            - observed_array
        )

        scale = np.maximum(
            np.maximum(
                np.abs(expected_array),
                np.abs(observed_array),
            ),
            np.finfo(np.float64).tiny,
        )

        relative_error = (
            absolute_error / scale
        )

        passed = bool(
            np.allclose(
                expected_array,
                observed_array,
                rtol=1e-12,
                atol=0.0,
            )
        )

        maximum_relative_error = float(
            relative_error.max()
        )
    else:
        passed = bool(
            expected_values.equals(
                observed_values
            )
        )

        maximum_relative_error = 0.0

    sampled_cleaning_records.append(
        {
            "column": column,
            "passed": passed,
            "maximum_relative_error": (
                maximum_relative_error
            ),
        }
    )

sampled_cleaning_table = pd.DataFrame(
    sampled_cleaning_records
)

display(sampled_cleaning_table)

assert sampled_cleaning_table[
    "passed"
].all()

sampled_cleaning_passed = True

print("SAMPLED 11-TO-9 TRANSFORMATION VERIFIED")

In [ ]:
full_identity_hashes = (
    pd.util.hash_pandas_object(
        processed_normalized[
            RECORD_KEY_COLUMNS
        ],
        index=False,
        categorize=True,
    )
)

sample_identity_hashes = (
    pd.util.hash_pandas_object(
        sampled_normalized[
            RECORD_KEY_COLUMNS
        ],
        index=False,
        categorize=True,
    )
)

full_identity_counts = (
    full_identity_hashes
    .value_counts()
)

sample_identity_counts = (
    sample_identity_hashes
    .value_counts()
)

available_full_counts = (
    full_identity_counts.reindex(
        sample_identity_counts.index,
        fill_value=0,
    )
)

sample_identity_subset = bool(
    (
        sample_identity_counts
        <= available_full_counts
    ).all()
)

unmatched_sample_identity_keys = int(
    (
        sample_identity_counts
        > available_full_counts
    ).sum()
)

print(
    "Sample identity multiset is a "
    "subset of full data:",
    sample_identity_subset,
)

print(
    "Unmatched sampled identity keys:",
    unmatched_sample_identity_keys,
)

assert sample_identity_subset
assert unmatched_sample_identity_keys == 0

In [ ]:
full_key_counts_direct = (
    processed_normalized
    .groupby(
        RECORD_KEY_COLUMNS,
        dropna=False,
        sort=False,
    )
    .size()
    .rename("full_count")
)

sample_key_counts_direct = (
    sampled_normalized
    .groupby(
        RECORD_KEY_COLUMNS,
        dropna=False,
        sort=False,
    )
    .size()
    .rename("sample_count")
)

direct_key_comparison = (
    pd.concat(
        [
            full_key_counts_direct,
            sample_key_counts_direct,
        ],
        axis=1,
    )
    .fillna(0)
    .astype(
        {
            "full_count": np.int64,
            "sample_count": np.int64,
        }
    )
    .reset_index()
)

direct_key_comparison[
    "sample_minus_full"
] = (
    direct_key_comparison[
        "sample_count"
    ]
    - direct_key_comparison[
        "full_count"
    ]
)

direct_key_comparison[
    "excess_sample_rows"
] = (
    direct_key_comparison[
        "sample_minus_full"
    ]
    .clip(lower=0)
)

sample_keys_absent_from_full = (
    direct_key_comparison.loc[
        (
            direct_key_comparison[
                "sample_count"
            ]
            > 0
        )
        & (
            direct_key_comparison[
                "full_count"
            ]
            == 0
        )
    ]
    .copy()
)

sample_keys_exceeding_full = (
    direct_key_comparison.loc[
        direct_key_comparison[
            "sample_count"
        ]
        > direct_key_comparison[
            "full_count"
        ]
    ]
    .copy()
)

print(
    "Direct sample keys absent from full:",
    len(sample_keys_absent_from_full),
)

print(
    "Direct sample keys exceeding "
    "full multiplicity:",
    len(sample_keys_exceeding_full),
)

print(
    "Total excess sampled rows:",
    int(
        sample_keys_exceeding_full[
            "excess_sample_rows"
        ].sum()
    ),
)

print(
    "Total sampled rows represented:",
    int(
        direct_key_comparison[
            "sample_count"
        ].sum()
    ),
)

In [ ]:
display(
    sample_keys_exceeding_full[
        RECORD_KEY_COLUMNS
        + [
            "full_count",
            "sample_count",
            "sample_minus_full",
        ]
    ]
    .sort_values(
        [
            "sample_minus_full",
            "sample_count",
        ],
        ascending=False,
    )
    .head(30)
)

In [ ]:
full_identity_index = (
    processed_normalized[
        RECORD_KEY_COLUMNS
    ]
    .drop_duplicates()
)

sample_identity_index = (
    sampled_normalized[
        RECORD_KEY_COLUMNS
    ]
    .drop_duplicates()
)

identity_presence_check = (
    sample_identity_index.merge(
        full_identity_index.assign(
            present_in_full=True
        ),
        on=RECORD_KEY_COLUMNS,
        how="left",
        validate="one_to_one",
    )
)

identity_presence_check[
    "present_in_full"
] = (
    identity_presence_check[
        "present_in_full"
    ]
    .fillna(False)
    .astype(bool)
)

absent_unique_identities = (
    identity_presence_check.loc[
        ~identity_presence_check[
            "present_in_full"
        ]
    ]
    .copy()
)

print(
    "Unique sampled identities:",
    len(sample_identity_index),
)

print(
    "Unique sampled identities absent "
    "from the full dataset:",
    len(absent_unique_identities),
)

if len(absent_unique_identities) > 0:
    display(
        absent_unique_identities.head(30)
    )

In [ ]:
PEAK_EVENT_KEY = [
    "peak_time",
    "peak_time_ns",
    "ml_label",
]

full_peak_identities = (
    processed_normalized[
        PEAK_EVENT_KEY
    ]
    .drop_duplicates()
)

sample_peak_identities = (
    sampled_normalized[
        PEAK_EVENT_KEY
    ]
    .drop_duplicates()
)

peak_presence_check = (
    sample_peak_identities.merge(
        full_peak_identities.assign(
            peak_present_in_full=True
        ),
        on=PEAK_EVENT_KEY,
        how="left",
        validate="one_to_one",
    )
)

peak_presence_check[
    "peak_present_in_full"
] = (
    peak_presence_check[
        "peak_present_in_full"
    ]
    .fillna(False)
    .astype(bool)
)

absent_peak_identities = (
    peak_presence_check.loc[
        ~peak_presence_check[
            "peak_present_in_full"
        ]
    ]
)

print(
    "Unique sampled peak-event identities:",
    len(sample_peak_identities),
)

print(
    "Sampled peak-event identities absent "
    "from full data:",
    len(absent_peak_identities),
)

if len(absent_peak_identities) > 0:
    display(
        absent_peak_identities.head(30)
    )

In [ ]:
def make_comparison_representation(frame):
    comparison = frame[
        ENRICHED_COLUMNS
    ].copy()

    for column in INTEGER_COLUMNS:
        comparison[column] = (
            pd.to_numeric(
                comparison[column],
                errors="raise",
            )
            .round()
            .astype(np.int64)
        )

    for column in FLOAT_COLUMNS:
        comparison[column] = (
            pd.to_numeric(
                comparison[column],
                errors="raise",
            )
            .astype(np.float64)
            .round(12)
        )

    comparison["ml_label"] = (
        comparison["ml_label"]
        .astype(str)
    )

    return comparison


full_comparison = (
    make_comparison_representation(
        processed_normalized
    )
)

sample_comparison = (
    make_comparison_representation(
        sampled_normalized
    )
)

In [ ]:
full_row_counts_direct = (
    full_comparison
    .groupby(
        ENRICHED_COLUMNS,
        dropna=False,
        sort=False,
    )
    .size()
    .rename("full_count")
)

sample_row_counts_direct = (
    sample_comparison
    .groupby(
        ENRICHED_COLUMNS,
        dropna=False,
        sort=False,
    )
    .size()
    .rename("sample_count")
)

direct_row_comparison = (
    pd.concat(
        [
            full_row_counts_direct,
            sample_row_counts_direct,
        ],
        axis=1,
    )
    .fillna(0)
    .astype(
        {
            "full_count": np.int64,
            "sample_count": np.int64,
        }
    )
    .reset_index()
)

direct_row_comparison[
    "sample_minus_full"
] = (
    direct_row_comparison[
        "sample_count"
    ]
    - direct_row_comparison[
        "full_count"
    ]
)

row_keys_absent_from_full = (
    direct_row_comparison.loc[
        (
            direct_row_comparison[
                "sample_count"
            ]
            > 0
        )
        & (
            direct_row_comparison[
                "full_count"
            ]
            == 0
        )
    ]
)

row_keys_exceeding_full = (
    direct_row_comparison.loc[
        direct_row_comparison[
            "sample_count"
        ]
        > direct_row_comparison[
            "full_count"
        ]
    ]
)

print(
    "Rounded full-row identities absent "
    "from full data:",
    len(row_keys_absent_from_full),
)

print(
    "Rounded full-row identities exceeding "
    "full multiplicity:",
    len(row_keys_exceeding_full),
)

print(
    "Total excess rounded sampled rows:",
    int(
        (
            row_keys_exceeding_full[
                "sample_count"
            ]
            - row_keys_exceeding_full[
                "full_count"
            ]
        ).sum()
    ),
)

In [ ]:
full_unique_rows = (
    full_comparison
    .drop_duplicates()
)

sample_unique_rows = (
    sample_comparison
    .drop_duplicates()
)

unique_row_presence = (
    sample_unique_rows.merge(
        full_unique_rows.assign(
            present_in_full=True
        ),
        on=ENRICHED_COLUMNS,
        how="left",
        validate="one_to_one",
    )
)

unique_row_presence[
    "present_in_full"
] = (
    unique_row_presence[
        "present_in_full"
    ]
    .fillna(False)
    .astype(bool)
)

unique_sample_rows_absent = (
    unique_row_presence.loc[
        ~unique_row_presence[
            "present_in_full"
        ]
    ]
)

print(
    "Unique sampled full rows:",
    len(sample_unique_rows),
)

print(
    "Unique sampled full rows absent "
    "from full data:",
    len(unique_sample_rows_absent),
)

if len(unique_sample_rows_absent) > 0:
    display(
        unique_sample_rows_absent.head(20)
    )

In [ ]:
sample_subset_diagnostic = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "sample_rows": int(
        len(sampled_normalized)
    ),
    "full_rows": int(
        len(processed_normalized)
    ),
    "hash_based_exceeding_keys": int(
        unmatched_sample_identity_keys
    ),
    "direct_identity_keys_absent": int(
        len(
            sample_keys_absent_from_full
        )
    ),
    "direct_identity_keys_exceeding_multiplicity": int(
        len(
            sample_keys_exceeding_full
        )
    ),
    "direct_excess_sample_rows": int(
        sample_keys_exceeding_full[
            "excess_sample_rows"
        ].sum()
    ),
    "unique_identity_keys_absent": int(
        len(absent_unique_identities)
    ),
    "unique_peak_event_keys_absent": int(
        len(absent_peak_identities)
    ),
    "rounded_full_row_keys_absent": int(
        len(row_keys_absent_from_full)
    ),
    "rounded_full_row_keys_exceeding_multiplicity": int(
        len(row_keys_exceeding_full)
    ),
    "unique_rounded_sample_rows_absent": int(
        len(unique_sample_rows_absent)
    ),
}

SAMPLE_SUBSET_DIAGNOSTIC_JSON = (
    OUTPUT_DIR
    / "sample_subset_investigation.json"
)

SAMPLE_SUBSET_DIAGNOSTIC_JSON.write_text(
    json.dumps(
        sample_subset_diagnostic,
        indent=2,
        allow_nan=False,
    )
    + "\n"
)

SAMPLE_KEY_DIAGNOSTIC_CSV = (
    OUTPUT_DIR
    / "sample_excess_identity_keys.csv"
)

sample_keys_exceeding_full.to_csv(
    SAMPLE_KEY_DIAGNOSTIC_CSV,
    index=False,
)

print("SAVED:")
print(SAMPLE_SUBSET_DIAGNOSTIC_JSON)
print(SAMPLE_KEY_DIAGNOSTIC_CSV)
print()
print("SAMPLE SUBSET INVESTIGATION COMPLETE")

## Step 25B — Diagnose repeated sampled records by class

In [ ]:
# Aggregate the multiplicity excess by class.

excess_by_class = (
    sample_keys_exceeding_full
    .groupby("ml_label", as_index=False)
    .agg(
        identities_exceeding_full=(
            "excess_sample_rows",
            "size",
        ),
        excess_sample_rows=(
            "excess_sample_rows",
            "sum",
        ),
        full_rows_for_these_identities=(
            "full_count",
            "sum",
        ),
        sampled_rows_for_these_identities=(
            "sample_count",
            "sum",
        ),
    )
    .sort_values(
        [
            "excess_sample_rows",
            "identities_exceeding_full",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

print("Multiplicity excess by class:")
display(excess_by_class)

print()
print(
    "Classes with excess sampled multiplicity:",
    len(excess_by_class),
)

print(
    "Total identities exceeding full multiplicity:",
    int(
        excess_by_class[
            "identities_exceeding_full"
        ].sum()
    ),
)

print(
    "Total excess sampled rows:",
    int(
        excess_by_class[
            "excess_sample_rows"
        ].sum()
    ),
)

assert (
    excess_by_class[
        "identities_exceeding_full"
    ].sum()
    == 132
)

assert (
    excess_by_class[
        "excess_sample_rows"
    ].sum()
    == 213
)

print()
print("CLASS-LEVEL EXCESS CHECK PASSED")

In [ ]:
full_class_counts = (
    original_data["ml_label"]
    .value_counts()
    .rename("full_rows")
)

sample_class_counts = (
    sampled_data["ml_label"]
    .value_counts()
    .rename("sample_rows")
)

class_sampling_summary = (
    pd.concat(
        [
            full_class_counts,
            sample_class_counts,
        ],
        axis=1,
    )
    .fillna(0)
    .astype(int)
    .reset_index()
    .rename(
        columns={
            "index": "ml_label",
        }
    )
)

class_sampling_summary[
    "full_fraction"
] = (
    class_sampling_summary["full_rows"]
    / len(original_data)
)

class_sampling_summary[
    "sample_fraction"
] = (
    class_sampling_summary["sample_rows"]
    / len(sampled_data)
)

class_sampling_summary[
    "sample_to_full_fraction_ratio"
] = (
    class_sampling_summary["sample_fraction"]
    / class_sampling_summary["full_fraction"]
)

class_sampling_summary = (
    class_sampling_summary
    .merge(
        excess_by_class[
            [
                "ml_label",
                "identities_exceeding_full",
                "excess_sample_rows",
            ]
        ],
        on="ml_label",
        how="left",
    )
    .fillna(
        {
            "identities_exceeding_full": 0,
            "excess_sample_rows": 0,
        }
    )
)

class_sampling_summary[
    "identities_exceeding_full"
] = (
    class_sampling_summary[
        "identities_exceeding_full"
    ]
    .astype(int)
)

class_sampling_summary[
    "excess_sample_rows"
] = (
    class_sampling_summary[
        "excess_sample_rows"
    ]
    .astype(int)
)

class_sampling_summary = (
    class_sampling_summary
    .sort_values(
        "sample_to_full_fraction_ratio",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(class_sampling_summary)

print()
print(
    "Full rows:",
    int(class_sampling_summary["full_rows"].sum()),
)

print(
    "Sample rows:",
    int(class_sampling_summary["sample_rows"].sum()),
)

assert (
    class_sampling_summary["full_rows"].sum()
    == len(original_data)
)

assert (
    class_sampling_summary["sample_rows"].sum()
    == len(sampled_data)
)

print()
print("CLASS-DISTRIBUTION CHECK PASSED")

In [ ]:
sample_unique_rows = (
    sampled_data
    .drop_duplicates()
    .shape[0]
)

sample_duplicate_occurrences = (
    len(sampled_data)
    - sample_unique_rows
)

full_unique_rows = (
    original_data
    .drop_duplicates()
    .shape[0]
)

full_duplicate_occurrences = (
    len(original_data)
    - full_unique_rows
)

duplicate_summary = {
    "full_rows": int(len(original_data)),
    "full_unique_rows": int(full_unique_rows),
    "full_duplicate_occurrences": int(
        full_duplicate_occurrences
    ),
    "sample_rows": int(len(sampled_data)),
    "sample_unique_rows": int(
        sample_unique_rows
    ),
    "sample_duplicate_occurrences": int(
        sample_duplicate_occurrences
    ),
    "identities_exceeding_full_multiplicity": int(
        len(sample_keys_exceeding_full)
    ),
    "excess_sample_rows": int(
        sample_keys_exceeding_full[
            "excess_sample_rows"
        ].sum()
    ),
}

print(
    json.dumps(
        duplicate_summary,
        indent=2,
        allow_nan=False,
    )
)

In [ ]:
CLASS_SAMPLING_CSV = (
    OUTPUT_DIR
    / "sample_class_distribution_audit.csv"
)

EXCESS_BY_CLASS_CSV = (
    OUTPUT_DIR
    / "sample_multiplicity_excess_by_class.csv"
)

SAMPLING_AUDIT_JSON = (
    OUTPUT_DIR
    / "sampling_provenance_audit.json"
)

class_sampling_summary.to_csv(
    CLASS_SAMPLING_CSV,
    index=False,
)

excess_by_class.to_csv(
    EXCESS_BY_CLASS_CSV,
    index=False,
)

sampling_audit = {
    "full_rows": int(len(original_data)),
    "sample_rows": int(len(sampled_data)),
    "unique_sample_identity_keys": 49039,
    "unique_sample_identity_keys_absent_from_full": 0,
    "unique_sample_peak_event_keys": 49038,
    "sample_peak_event_keys_absent_from_full": 0,
    "unique_sample_full_rows_absent_from_full": 0,
    "identities_exceeding_full_multiplicity": 132,
    "total_excess_sample_rows": 213,
    "sample_unique_rows": int(
        sample_unique_rows
    ),
    "sample_duplicate_occurrences": int(
        sample_duplicate_occurrences
    ),
    "interpretation": (
        "Every unique sampled record is present in the "
        "full processed dataset, but some records occur "
        "more frequently in the sampled dataset than in "
        "the full dataset. The sampled dataset is therefore "
        "not a strict sample without replacement and likely "
        "contains repeated selections introduced during "
        "class-wise sampling or oversampling."
    ),
}

SAMPLING_AUDIT_JSON.write_text(
    json.dumps(
        sampling_audit,
        indent=2,
        allow_nan=False,
    )
    + "\n"
)

print("SAVED:")
print(CLASS_SAMPLING_CSV)
print(EXCESS_BY_CLASS_CSV)
print(SAMPLING_AUDIT_JSON)
print()
print("STEP 25 SAMPLING AUDIT COMPLETE")

## Step 26 — Locate and verify the historical sampling implementation

In [ ]:
from pathlib import Path
import re
import json
import hashlib

PROJECT_ROOT = Path(
    str(PORTABLE_PROJECT_ROOT)
)

SEARCH_EXTENSIONS = {
    ".py",
    ".ipynb",
    ".md",
    ".txt",
    ".log",
}

SEARCH_TERMS = [
    "sampled_data.csv",
    "sampled_data_cleaned.csv",
    "sample(",
    "replace=True",
    "replace=False",
    "frac=0.1",
    "frac = 0.1",
    "n_samples",
    "resample(",
    "groupby",
    "ml_label",
]

excluded_directory_names = {
    ".git",
    "__pycache__",
    ".ipynb_checkpoints",
    "node_modules",
    "site-packages",
    "pilot_outputs",
    "targeted_ablation_v1",
    "feature_provenance_v1",
}

candidate_files = []

for path in PROJECT_ROOT.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() not in SEARCH_EXTENSIONS:
        continue

    if any(
        part in excluded_directory_names
        for part in path.parts
    ):
        continue

    try:
        text = path.read_text(
            encoding="utf-8",
            errors="ignore",
        )
    except Exception:
        continue

    matching_terms = [
        term
        for term in SEARCH_TERMS
        if term.lower() in text.lower()
    ]

    if matching_terms:
        candidate_files.append(
            {
                "path": str(path),
                "matching_terms": matching_terms,
                "bytes": path.stat().st_size,
            }
        )

print(
    "Candidate files containing sampling-related terms:",
    len(candidate_files),
)

for index, result in enumerate(
    candidate_files,
    start=1,
):
    print()
    print(f"[{index}] {result['path']}")
    print(
        "    matches:",
        result["matching_terms"],
    )
    print(
        "    bytes:",
        result["bytes"],
    )

In [ ]:
sampling_line_patterns = [
    re.compile(
        r"sampled_data",
        re.IGNORECASE,
    ),
    re.compile(
        r"\.sample\s*\(",
        re.IGNORECASE,
    ),
    re.compile(
        r"resample\s*\(",
        re.IGNORECASE,
    ),
    re.compile(
        r"replace\s*=",
        re.IGNORECASE,
    ),
    re.compile(
        r"frac\s*=",
        re.IGNORECASE,
    ),
]

sampling_evidence = []

for result in candidate_files:
    path = Path(result["path"])

    try:
        lines = path.read_text(
            encoding="utf-8",
            errors="ignore",
        ).splitlines()
    except Exception:
        continue

    matched_line_numbers = []

    for line_number, line in enumerate(
        lines,
        start=1,
    ):
        if any(
            pattern.search(line)
            for pattern in sampling_line_patterns
        ):
            matched_line_numbers.append(
                line_number
            )

    if not matched_line_numbers:
        continue

    print()
    print("=" * 80)
    print(path)
    print("=" * 80)

    displayed = set()

    for line_number in matched_line_numbers:
        start = max(1, line_number - 4)
        stop = min(
            len(lines),
            line_number + 4,
        )

        for current in range(start, stop + 1):
            if current in displayed:
                continue

            print(
                f"{current:6d}: "
                f"{lines[current - 1]}"
            )
            displayed.add(current)

        print()

    sampling_evidence.append(
        {
            "path": str(path),
            "matched_line_numbers": (
                matched_line_numbers
            ),
        }
    )

print()
print(
    "Files with direct sampling evidence:",
    len(sampling_evidence),
)

In [ ]:
SAMPLING_CODE_SEARCH_JSON = (
    OUTPUT_DIR
    / "historical_sampling_code_search.json"
)

sampling_code_report = {
    "project_root": str(PROJECT_ROOT),
    "candidate_file_count": len(
        candidate_files
    ),
    "evidence_file_count": len(
        sampling_evidence
    ),
    "candidate_files": candidate_files,
    "sampling_evidence": sampling_evidence,
}

SAMPLING_CODE_SEARCH_JSON.write_text(
    json.dumps(
        sampling_code_report,
        indent=2,
        allow_nan=False,
    )
    + "\n"
)

print("SAVED:")
print(SAMPLING_CODE_SEARCH_JSON)
print()
print("STEP 26–28 CODE SEARCH COMPLETE")

## Step 29 — Extract the historical sample-construction code

In [ ]:
import json
import hashlib
from pathlib import Path

SAMPLING_CANDIDATES = [
    PROJECT_ROOT
    / "Round3"
    / "sample_data_creator.ipynb",

    PROJECT_ROOT
    / "Round3"
    / "light_resampling.ipynb",

    PROJECT_ROOT
    / "Round3"
    / "sampling_test.ipynb",

    PROJECT_ROOT
    / "Round2"
    / "Data_Exploration.ipynb",
]

TARGET_PATTERNS = [
    "sample(",
    ".sample(",
    "resample(",
    "replace=",
    "replace =",
    "frac=",
    "frac =",
    "n_samples",
    "groupby(",
    "ml_label",
    "to_csv(",
    "sampled_data",
    "concat(",
]


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


extracted_sampling_cells = []

for notebook_path in SAMPLING_CANDIDATES:
    print()
    print("=" * 100)
    print("NOTEBOOK:", notebook_path)
    print("=" * 100)

    if not notebook_path.exists():
        print("STATUS: NOT FOUND")
        continue

    notebook = json.loads(
        notebook_path.read_text(
            encoding="utf-8",
            errors="ignore",
        )
    )

    print("SHA-256:", sha256_file(notebook_path))
    print(
        "Notebook cells:",
        len(notebook.get("cells", [])),
    )

    matching_cell_count = 0

    for cell_number, cell in enumerate(
        notebook.get("cells", []),
        start=1,
    ):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(
            cell.get("source", [])
        )

        matched_patterns = [
            pattern
            for pattern in TARGET_PATTERNS
            if pattern.lower() in source.lower()
        ]

        if not matched_patterns:
            continue

        matching_cell_count += 1

        print()
        print("-" * 100)
        print(
            f"CODE CELL {cell_number}; "
            f"matches: {matched_patterns}"
        )
        print("-" * 100)
        print(source.rstrip())

        extracted_sampling_cells.append(
            {
                "notebook": str(notebook_path),
                "notebook_sha256": (
                    sha256_file(notebook_path)
                ),
                "cell_number": cell_number,
                "matched_patterns": (
                    matched_patterns
                ),
                "source": source,
            }
        )

    print()
    print(
        "Matching code cells:",
        matching_cell_count,
    )

print()
print(
    "TOTAL EXTRACTED CODE CELLS:",
    len(extracted_sampling_cells),
)

In [ ]:
HISTORICAL_SAMPLING_EVIDENCE_JSON = (
    OUTPUT_DIR
    / "historical_sampling_code_evidence.json"
)

historical_sampling_evidence = {
    "candidate_notebooks": [
        str(path)
        for path in SAMPLING_CANDIDATES
    ],
    "extracted_cell_count": len(
        extracted_sampling_cells
    ),
    "cells": extracted_sampling_cells,
}

HISTORICAL_SAMPLING_EVIDENCE_JSON.write_text(
    json.dumps(
        historical_sampling_evidence,
        indent=2,
        allow_nan=False,
    )
    + "\n"
)

print("SAVED:")
print(
    HISTORICAL_SAMPLING_EVIDENCE_JSON
)
print()
print(
    "STEP 29–30 HISTORICAL CODE "
    "EXTRACTION COMPLETE"
)

## Step 31 — Reconstruct the historical 50,000-row stratified sample

In [ ]:
from sklearn.model_selection import train_test_split

historical_50k, historical_remainder = (
    train_test_split(
        original_data,
        stratify=original_data["ml_label"],
        train_size=50_000,
        random_state=42,
    )
)

historical_50k = (
    historical_50k
    .reset_index(drop=True)
)

historical_remainder = (
    historical_remainder
    .reset_index(drop=True)
)

historical_50k_counts = (
    historical_50k["ml_label"]
    .value_counts()
    .rename("reconstructed_50k_rows")
)

observed_sample_counts = (
    sampled_data["ml_label"]
    .value_counts()
    .rename("observed_sample_rows")
)

full_counts = (
    original_data["ml_label"]
    .value_counts()
    .rename("full_rows")
)

historical_count_comparison = (
    pd.concat(
        [
            full_counts,
            historical_50k_counts,
            observed_sample_counts,
        ],
        axis=1,
    )
    .fillna(0)
    .astype(int)
    .reset_index()
    .rename(
        columns={
            "index": "ml_label",
        }
    )
)

historical_count_comparison[
    "observed_minus_reconstructed"
] = (
    historical_count_comparison[
        "observed_sample_rows"
    ]
    - historical_count_comparison[
        "reconstructed_50k_rows"
    ]
)

historical_count_comparison = (
    historical_count_comparison
    .sort_values(
        "observed_minus_reconstructed",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(historical_count_comparison)

print()
print(
    "Reconstructed rows:",
    len(historical_50k),
)

print(
    "Observed sampled rows:",
    len(sampled_data),
)

print(
    "Net additional rows:",
    len(sampled_data)
    - len(historical_50k),
)

assert len(historical_50k) == 50_000
assert len(sampled_data) == 50_554

print()
print("INITIAL 50K RECONSTRUCTION COMPLETE")

In [ ]:
RARE_CLASSES = [
    "None_of_the_Above",
    "Helix",
    "Chirp",
    "Wandering_Line",
]

COMMON_CLASSES = sorted(
    set(original_data["ml_label"].unique())
    - set(RARE_CLASSES)
)

historical_common = (
    historical_50k.loc[
        historical_50k["ml_label"].isin(
            COMMON_CLASSES
        )
    ]
    .copy()
)

observed_common = (
    sampled_data.loc[
        sampled_data["ml_label"].isin(
            COMMON_CLASSES
        )
    ]
    .copy()
)

print(
    "Reconstructed common-class rows:",
    len(historical_common),
)

print(
    "Observed common-class rows:",
    len(observed_common),
)

FULL_ROW_COLUMNS = (
    original_data.columns.tolist()
)

historical_common_hashes = (
    pd.util.hash_pandas_object(
        historical_common[
            FULL_ROW_COLUMNS
        ],
        index=False,
        categorize=True,
    )
    .value_counts()
    .sort_index()
)

observed_common_hashes = (
    pd.util.hash_pandas_object(
        observed_common[
            FULL_ROW_COLUMNS
        ],
        index=False,
        categorize=True,
    )
    .value_counts()
    .sort_index()
)

common_exact_multiset_match = (
    historical_common_hashes.equals(
        observed_common_hashes
    )
)

print(
    "Exact common-class row-multiset match:",
    common_exact_multiset_match,
)

common_class_count_match = (
    historical_common[
        "ml_label"
    ]
    .value_counts()
    .sort_index()
    .equals(
        observed_common[
            "ml_label"
        ]
        .value_counts()
        .sort_index()
    )
)

print(
    "Common-class count match:",
    common_class_count_match,
)

assert common_class_count_match

print()
print("COMMON-CLASS RECONSTRUCTION CHECK COMPLETE")

In [ ]:
rare_reconstruction_rows = []

for label in RARE_CLASSES:
    full_class = (
        original_data.loc[
            original_data["ml_label"]
            == label
        ]
        .copy()
    )

    observed_class = (
        sampled_data.loc[
            sampled_data["ml_label"]
            == label
        ]
        .copy()
    )

    reconstructed_50k_class = (
        historical_50k.loc[
            historical_50k["ml_label"]
            == label
        ]
        .copy()
    )

    rare_reconstruction_rows.append(
        {
            "ml_label": label,
            "full_rows": len(full_class),
            "initial_50k_rows": len(
                reconstructed_50k_class
            ),
            "observed_sample_rows": len(
                observed_class
            ),
            "observed_equals_full_count": (
                len(observed_class)
                == len(full_class)
            ),
            "net_rows_added": (
                len(observed_class)
                - len(reconstructed_50k_class)
            ),
            "observed_unique_rows": (
                len(
                    observed_class
                    .drop_duplicates()
                )
            ),
            "full_unique_rows": (
                len(
                    full_class
                    .drop_duplicates()
                )
            ),
        }
    )

rare_reconstruction_summary = (
    pd.DataFrame(
        rare_reconstruction_rows
    )
)

display(rare_reconstruction_summary)

print()
print(
    "Total initial rare-class rows:",
    int(
        rare_reconstruction_summary[
            "initial_50k_rows"
        ].sum()
    ),
)

print(
    "Total observed rare-class rows:",
    int(
        rare_reconstruction_summary[
            "observed_sample_rows"
        ].sum()
    ),
)

print(
    "Total net rare-class additions:",
    int(
        rare_reconstruction_summary[
            "net_rows_added"
        ].sum()
    ),
)

assert (
    rare_reconstruction_summary[
        "observed_equals_full_count"
    ].all()
)

assert (
    rare_reconstruction_summary[
        "net_rows_added"
    ].sum()
    == 554
)

print()
print(
    "RARE-CLASS COUNT RECONSTRUCTION PASSED"
)

In [ ]:
HISTORICAL_COUNT_COMPARISON_CSV = (
    OUTPUT_DIR
    / "historical_50k_count_reconstruction.csv"
)

RARE_RECONSTRUCTION_CSV = (
    OUTPUT_DIR
    / "historical_rare_class_reconstruction.csv"
)

HISTORICAL_RECONSTRUCTION_JSON = (
    OUTPUT_DIR
    / "historical_sampling_reconstruction.json"
)

historical_count_comparison.to_csv(
    HISTORICAL_COUNT_COMPARISON_CSV,
    index=False,
)

rare_reconstruction_summary.to_csv(
    RARE_RECONSTRUCTION_CSV,
    index=False,
)

historical_reconstruction_report = {
    "initial_stratified_rows": int(
        len(historical_50k)
    ),
    "final_sampled_rows": int(
        len(sampled_data)
    ),
    "net_rows_added": int(
        len(sampled_data)
        - len(historical_50k)
    ),
    "rare_classes": RARE_CLASSES,
    "initial_rare_rows": int(
        rare_reconstruction_summary[
            "initial_50k_rows"
        ].sum()
    ),
    "final_rare_rows": int(
        rare_reconstruction_summary[
            "observed_sample_rows"
        ].sum()
    ),
    "rare_rows_added": int(
        rare_reconstruction_summary[
            "net_rows_added"
        ].sum()
    ),
    "common_class_count_match": bool(
        common_class_count_match
    ),
    "common_exact_multiset_match": bool(
        common_exact_multiset_match
    ),
    "interpretation": (
        "The historical sampled dataset began as a "
        "50,000-row stratified sample with random_state=42. "
        "The four rarest classes were subsequently expanded "
        "from their stratified-sample counts to counts equal "
        "to their complete-data class sizes, producing "
        "50,554 rows."
    ),
}

HISTORICAL_RECONSTRUCTION_JSON.write_text(
    json.dumps(
        historical_reconstruction_report,
        indent=2,
        allow_nan=False,
    )
    + "\n"
)

print("SAVED:")
print(
    HISTORICAL_COUNT_COMPARISON_CSV
)
print(RARE_RECONSTRUCTION_CSV)
print(HISTORICAL_RECONSTRUCTION_JSON)
print()
print(
    "STEP 31–34 HISTORICAL "
    "RECONSTRUCTION COMPLETE"
)

## Step 35 — Finalize the historical sampling-provenance conclusion

In [ ]:
FINAL_SAMPLING_AUDIT_JSON = (
    OUTPUT_DIR
    / "historical_sampling_final_audit.json"
)

final_sampling_audit = {
    "status": (
        "resolved_at_class_count_level"
    ),
    "full_dataset_rows": int(
        len(original_data)
    ),
    "historical_initial_sample_rows": 50_000,
    "historical_final_sample_rows": int(
        len(sampled_data)
    ),
    "larger_class_count_match": bool(
        common_class_count_match
    ),
    "larger_class_exact_row_multiset_match": bool(
        common_exact_multiset_match
    ),
    "rare_classes": {
        row["ml_label"]: {
            "full_rows": int(
                row["full_rows"]
            ),
            "initial_stratified_rows": int(
                row["initial_50k_rows"]
            ),
            "final_sample_rows": int(
                row["observed_sample_rows"]
            ),
            "net_rows_added": int(
                row["net_rows_added"]
            ),
            "observed_unique_rows": int(
                row["observed_unique_rows"]
            ),
            "full_unique_rows": int(
                row["full_unique_rows"]
            ),
        }
        for row in (
            rare_reconstruction_summary
            .to_dict("records")
        )
    },
    "initial_combined_rare_rows": int(
        rare_reconstruction_summary[
            "initial_50k_rows"
        ].sum()
    ),
    "final_combined_rare_rows": int(
        rare_reconstruction_summary[
            "observed_sample_rows"
        ].sum()
    ),
    "net_rows_added": int(
        rare_reconstruction_summary[
            "net_rows_added"
        ].sum()
    ),
    "unique_sample_records_absent_from_full": 0,
    "identities_exceeding_full_multiplicity": 132,
    "excess_repeated_rows": 213,
    "supported_conclusions": [
        (
            "The 20 larger classes have exactly the "
            "counts expected from a 50000-row "
            "stratified sample."
        ),
        (
            "The four rarest classes were expanded "
            "from 61 combined rows to 615 combined "
            "rows, producing the final total of "
            "50554 rows."
        ),
        (
            "Every unique sampled record originates "
            "from the full processed dataset."
        ),
        (
            "The rare-class expansion involved "
            "replacement-based repeated selections."
        ),
    ],
    "limitations": [
        (
            "The exact common-class row multiset was "
            "not reproduced from original_data.csv "
            "using train_test_split with "
            "random_state=42."
        ),
        (
            "The precise historical intermediate "
            "input ordering or source snapshot is "
            "therefore not recoverable from the "
            "currently archived files."
        ),
        (
            "The reconstruction supports the "
            "class-count procedure but not an exact "
            "row-for-row regeneration."
        ),
    ],
    "revision_protocol_note": (
        "The referee-driven revision experiments do "
        "not reuse the historical 50554-row sampling "
        "procedure. They use the reconstructed, "
        "duplicate-aware, group-disjoint O3 cohort "
        "and frozen random and chronological split "
        "assignments."
    ),
}

FINAL_SAMPLING_AUDIT_JSON.write_text(
    json.dumps(
        final_sampling_audit,
        indent=2,
        allow_nan=False,
    )
    + "\n"
)

print(
    json.dumps(
        final_sampling_audit,
        indent=2,
        allow_nan=False,
    )
)

print()
print("SAVED:")
print(FINAL_SAMPLING_AUDIT_JSON)
print()
print(
    "HISTORICAL SAMPLING AUDIT CLOSED"
)

## Step 36 — Audit redundancy in the original eleven-feature representation

In [ ]:
NUMERIC_FEATURES_11 = [
    "peak_time",
    "peak_time_ns",
    "start_time",
    "start_time_ns",
    "duration",
    "peak_frequency",
    "central_freq",
    "bandwidth",
    "amplitude",
    "snr",
    "q_value",
]

feature_frame_11 = (
    original_data[
        NUMERIC_FEATURES_11
    ]
    .copy()
)

assert len(feature_frame_11) == 500_524
assert not feature_frame_11.isna().any().any()

pearson_correlation = (
    feature_frame_11
    .corr(method="pearson")
)

spearman_correlation = (
    feature_frame_11
    .corr(method="spearman")
)

print("Pearson correlation matrix:")
display(
    pearson_correlation.round(6)
)

print()
print("Spearman correlation matrix:")
display(
    spearman_correlation.round(6)
)

pairwise_correlations = []

for index, feature_a in enumerate(
    NUMERIC_FEATURES_11
):
    for feature_b in (
        NUMERIC_FEATURES_11[index + 1:]
    ):
        pairwise_correlations.append(
            {
                "feature_a": feature_a,
                "feature_b": feature_b,
                "pearson": float(
                    pearson_correlation.loc[
                        feature_a,
                        feature_b,
                    ]
                ),
                "absolute_pearson": abs(
                    float(
                        pearson_correlation.loc[
                            feature_a,
                            feature_b,
                        ]
                    )
                ),
                "spearman": float(
                    spearman_correlation.loc[
                        feature_a,
                        feature_b,
                    ]
                ),
                "absolute_spearman": abs(
                    float(
                        spearman_correlation.loc[
                            feature_a,
                            feature_b,
                        ]
                    )
                ),
            }
        )

pairwise_correlation_table = (
    pd.DataFrame(
        pairwise_correlations
    )
    .sort_values(
        [
            "absolute_pearson",
            "absolute_spearman",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

print()
print("Strongest pairwise correlations:")
display(
    pairwise_correlation_table.head(20)
)

print()
print("PAIRWISE CORRELATION AUDIT COMPLETE")

In [ ]:
NANOSECONDS_PER_SECOND = 1_000_000_000

peak_time_seconds = (
    original_data["peak_time"]
    .astype("int64")
)

peak_time_nanoseconds = (
    original_data["peak_time_ns"]
    .astype("int64")
)

start_time_seconds = (
    original_data["start_time"]
    .astype("int64")
)

start_time_nanoseconds = (
    original_data["start_time_ns"]
    .astype("int64")
)

peak_gps_nanoseconds = (
    peak_time_seconds
    * NANOSECONDS_PER_SECOND
    + peak_time_nanoseconds
)

start_gps_nanoseconds = (
    start_time_seconds
    * NANOSECONDS_PER_SECOND
    + start_time_nanoseconds
)

peak_minus_start_seconds = (
    peak_gps_nanoseconds
    - start_gps_nanoseconds
) / NANOSECONDS_PER_SECOND

timestamp_relationship_summary = {
    "rows": int(len(original_data)),
    "start_time_equals_peak_time_rows": int(
        (
            start_time_seconds
            == peak_time_seconds
        ).sum()
    ),
    "start_time_equals_peak_time_fraction": float(
        (
            start_time_seconds
            == peak_time_seconds
        ).mean()
    ),
    "minimum_peak_minus_start_seconds": float(
        peak_minus_start_seconds.min()
    ),
    "median_peak_minus_start_seconds": float(
        peak_minus_start_seconds.median()
    ),
    "maximum_peak_minus_start_seconds": float(
        peak_minus_start_seconds.max()
    ),
}

print(
    json.dumps(
        timestamp_relationship_summary,
        indent=2,
        allow_nan=False,
    )
)

print()
print(
    "Unique integer-second differences "
    "(peak_time - start_time):"
)

integer_second_differences = (
    peak_time_seconds
    - start_time_seconds
)

display(
    integer_second_differences
    .value_counts()
    .sort_index()
    .rename_axis(
        "peak_time_minus_start_time"
    )
    .rename("rows")
    .reset_index()
    .head(30)
)

print()
print("ABSOLUTE TIMESTAMPS RECONSTRUCTED")

In [ ]:
duration = (
    original_data["duration"]
    .astype("float64")
)

timestamp_duration_tests = []

candidate_start_offsets = {
    "peak_minus_duration": duration,
    "peak_minus_half_duration": (
        duration / 2.0
    ),
    "peak_minus_quarter_duration": (
        duration / 4.0
    ),
}

for relationship, candidate_offset in (
    candidate_start_offsets.items()
):
    residual_seconds = (
        peak_minus_start_seconds
        - candidate_offset
    )

    absolute_residual = (
        residual_seconds.abs()
    )

    timestamp_duration_tests.append(
        {
            "relationship": relationship,
            "median_absolute_error_seconds": float(
                absolute_residual.median()
            ),
            "p95_absolute_error_seconds": float(
                absolute_residual.quantile(
                    0.95
                )
            ),
            "maximum_absolute_error_seconds": float(
                absolute_residual.max()
            ),
            "fraction_within_1_nanosecond": float(
                (
                    absolute_residual
                    <= 1e-9
                ).mean()
            ),
            "fraction_within_1_microsecond": float(
                (
                    absolute_residual
                    <= 1e-6
                ).mean()
            ),
            "fraction_within_1_millisecond": float(
                (
                    absolute_residual
                    <= 1e-3
                ).mean()
            ),
        }
    )

timestamp_duration_table = (
    pd.DataFrame(
        timestamp_duration_tests
    )
    .sort_values(
        "median_absolute_error_seconds"
    )
    .reset_index(drop=True)
)

display(timestamp_duration_table)

print()
print(
    "Correlation between trigger duration "
    "and peak-minus-start interval:"
)

print(
    "Pearson:",
    float(
        np.corrcoef(
            duration,
            peak_minus_start_seconds,
        )[0, 1]
    ),
)

print(
    "Spearman:",
    float(
        pd.Series(duration)
        .corr(
            pd.Series(
                peak_minus_start_seconds
            ),
            method="spearman",
        )
    ),
)

print()
print(
    "TIMESTAMP–DURATION RELATIONSHIP "
    "AUDIT COMPLETE"
)

In [ ]:
bandwidth = (
    original_data["bandwidth"]
    .astype("float64")
)

central_frequency = (
    original_data["central_freq"]
    .astype("float64")
)

peak_frequency = (
    original_data["peak_frequency"]
    .astype("float64")
)

q_value = (
    original_data["q_value"]
    .astype("float64")
)

valid_bandwidth = (
    np.isfinite(bandwidth)
    & np.isfinite(central_frequency)
    & np.isfinite(peak_frequency)
    & np.isfinite(q_value)
    & (bandwidth != 0)
    & (central_frequency != 0)
    & (peak_frequency != 0)
    & (q_value != 0)
)

bandwidth_relationship_candidates = {
    "central_freq_div_q": (
        central_frequency / q_value
    ),
    "peak_frequency_div_q": (
        peak_frequency / q_value
    ),
    "sqrt2_central_freq_div_q": (
        np.sqrt(2.0)
        * central_frequency
        / q_value
    ),
    "sqrt2_peak_frequency_div_q": (
        np.sqrt(2.0)
        * peak_frequency
        / q_value
    ),
    "central_freq_div_sqrt2_q": (
        central_frequency
        / (
            np.sqrt(2.0)
            * q_value
        )
    ),
    "peak_frequency_div_sqrt2_q": (
        peak_frequency
        / (
            np.sqrt(2.0)
            * q_value
        )
    ),
}

bandwidth_relationship_tests = []

for relationship, predicted_bandwidth in (
    bandwidth_relationship_candidates.items()
):
    observed = bandwidth.loc[
        valid_bandwidth
    ].to_numpy()

    predicted = (
        predicted_bandwidth.loc[
            valid_bandwidth
        ]
        .to_numpy()
    )

    absolute_error = np.abs(
        observed - predicted
    )

    relative_error = (
        absolute_error
        / np.maximum(
            np.abs(observed),
            np.finfo(float).tiny,
        )
    )

    bandwidth_relationship_tests.append(
        {
            "relationship": relationship,
            "rows": int(len(observed)),
            "median_absolute_error": float(
                np.median(
                    absolute_error
                )
            ),
            "maximum_absolute_error": float(
                np.max(
                    absolute_error
                )
            ),
            "median_relative_error": float(
                np.median(
                    relative_error
                )
            ),
            "p95_relative_error": float(
                np.quantile(
                    relative_error,
                    0.95,
                )
            ),
            "maximum_relative_error": float(
                np.max(
                    relative_error
                )
            ),
            "fraction_within_rtol_1e_minus_12": float(
                np.mean(
                    np.isclose(
                        observed,
                        predicted,
                        rtol=1e-12,
                        atol=0.0,
                    )
                )
            ),
            "fraction_within_rtol_1e_minus_6": float(
                np.mean(
                    np.isclose(
                        observed,
                        predicted,
                        rtol=1e-6,
                        atol=0.0,
                    )
                )
            ),
        }
    )

bandwidth_relationship_table = (
    pd.DataFrame(
        bandwidth_relationship_tests
    )
    .sort_values(
        [
            "median_relative_error",
            "p95_relative_error",
        ]
    )
    .reset_index(drop=True)
)

display(bandwidth_relationship_table)

print()
print(
    "Observed ratio summaries:"
)

ratio_summary = pd.DataFrame(
    {
        "bandwidth_times_q_div_central": (
            bandwidth
            * q_value
            / central_frequency
        ),
        "bandwidth_times_q_div_peak": (
            bandwidth
            * q_value
            / peak_frequency
        ),
        "central_div_bandwidth": (
            central_frequency
            / bandwidth
        ),
        "peak_div_bandwidth": (
            peak_frequency
            / bandwidth
        ),
    }
).describe(
    percentiles=[
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99,
    ]
)

display(ratio_summary)

print()
print(
    "BANDWIDTH RELATIONSHIP AUDIT COMPLETE"
)

## Step 40 — Save the feature-redundancy audit

In [ ]:
PAIRWISE_CORRELATION_CSV = (
    OUTPUT_DIR
    / "feature_pairwise_correlations.csv"
)

TIMESTAMP_DURATION_CSV = (
    OUTPUT_DIR
    / "timestamp_duration_relationships.csv"
)

BANDWIDTH_RELATIONSHIP_CSV = (
    OUTPUT_DIR
    / "bandwidth_relationship_tests.csv"
)

FEATURE_REDUNDANCY_JSON = (
    OUTPUT_DIR
    / "feature_redundancy_audit.json"
)

pairwise_correlation_table.to_csv(
    PAIRWISE_CORRELATION_CSV,
    index=False,
)

timestamp_duration_table.to_csv(
    TIMESTAMP_DURATION_CSV,
    index=False,
)

bandwidth_relationship_table.to_csv(
    BANDWIDTH_RELATIONSHIP_CSV,
    index=False,
)

feature_redundancy_audit = {
    "rows": int(len(original_data)),
    "removed_features": [
        "start_time",
        "bandwidth",
    ],
    "start_time_audit": {
        "pearson_with_peak_time": float(
            pearson_correlation.loc[
                "start_time",
                "peak_time",
            ]
        ),
        "spearman_with_peak_time": float(
            spearman_correlation.loc[
                "start_time",
                "peak_time",
            ]
        ),
        "equal_integer_second_rows": int(
            (
                original_data["start_time"]
                == original_data["peak_time"]
            ).sum()
        ),
        "equal_integer_second_fraction": float(
            (
                original_data["start_time"]
                == original_data["peak_time"]
            ).mean()
        ),
        "minimum_peak_minus_start_seconds": float(
            peak_minus_start_seconds.min()
        ),
        "median_peak_minus_start_seconds": float(
            peak_minus_start_seconds.median()
        ),
        "maximum_peak_minus_start_seconds": float(
            peak_minus_start_seconds.max()
        ),
        "interpretation": (
            "start_time is not algebraically identical "
            "to peak_time. Both encode nearly the same "
            "absolute observing-time axis, which produces "
            "a correlation rounded to unity over the O3 "
            "time range. Removing start_time reduces "
            "duplicated absolute-time information."
        ),
    },
    "bandwidth_audit": {
        "pearson_with_central_freq": float(
            pearson_correlation.loc[
                "bandwidth",
                "central_freq",
            ]
        ),
        "spearman_with_central_freq": float(
            spearman_correlation.loc[
                "bandwidth",
                "central_freq",
            ]
        ),
        "best_tested_formula": str(
            bandwidth_relationship_table.loc[
                0,
                "relationship",
            ]
        ),
        "best_formula_median_relative_error": float(
            bandwidth_relationship_table.loc[
                0,
                "median_relative_error",
            ]
        ),
        "best_formula_p95_relative_error": float(
            bandwidth_relationship_table.loc[
                0,
                "p95_relative_error",
            ]
        ),
        "interpretation": (
            "bandwidth is strongly associated with "
            "central_freq but is not reproduced by the "
            "tested frequency-and-Q identities. It is "
            "therefore highly correlated rather than "
            "algebraically redundant."
        ),
    },
    "corrected_feature_selection_statement": (
        "The initial catalog contained 11 numerical "
        "variables. start_time was excluded because it "
        "duplicated the absolute observing-time information "
        "already encoded by peak_time, and bandwidth was "
        "excluded because of its very strong association "
        "with central_freq. These exclusions reduced "
        "duplicated temporal information and "
        "multicollinearity; the variables are not "
        "algebraically identical."
    ),
    "richer_feature_decision": (
        "The original 11-feature representation should "
        "not be described as uniformly richer because "
        "start_time adds another absolute timestamp. "
        "Bandwidth is, however, a distinct spectral "
        "descriptor and can be evaluated separately as "
        "a targeted feature augmentation."
    ),
}

FEATURE_REDUNDANCY_JSON.write_text(
    json.dumps(
        feature_redundancy_audit,
        indent=2,
        allow_nan=False,
    )
    + "\n"
)

print(
    json.dumps(
        feature_redundancy_audit,
        indent=2,
        allow_nan=False,
    )
)

print()
print("SAVED:")
print(PAIRWISE_CORRELATION_CSV)
print(TIMESTAMP_DURATION_CSV)
print(BANDWIDTH_RELATIONSHIP_CSV)
print(FEATURE_REDUNDANCY_JSON)
print()
print("FEATURE-REDUNDANCY AUDIT CLOSED")